## 1. Environment check

Executer manuellement avec le kernel distant du PYNQ-Z2. Runtime attendu : tflite-runtime 2.13.0 recompile ARMv7-A / NEON / VFPv3, Python 3.10.4 et NumPy 1.21.5. Les chemins sont ceux du PYNQ. Aucune installation automatique.


In [1]:
import sys
import platform
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

print("Python version:", sys.version)
print("Architecture:", platform.machine())
print("Current working directory:", Path.cwd())
print("Home directory:", Path.home())

for package, label in [("numpy", "NumPy version"),
                       ("tflite-runtime", "tflite-runtime version")]:
    try:
        print(f"{label}: {version(package)}")
    except PackageNotFoundError:
        print(f"{label}: package non trouve dans ce kernel")


Python version: 3.10.4 (main, Apr  2 2022, 09:04:19) [GCC 11.2.0]
Architecture: armv7l
Current working directory: /home/xilinx/jupyter_notebooks
Home directory: /root
NumPy version: 1.21.5
tflite-runtime version: 2.13.0


## 2. Runtime/model loading

Charger une seule fois SSD MobileNet V1 et ses labels. L'entree uint8 et les quatre sorties conservent la signature de la baseline. Retirer la premiere entree ??? du label map si presente.


In [2]:
import numpy as np
from tflite_runtime.interpreter import Interpreter

print("NumPy import: OK —", np.__version__)
print("Interpreter import: OK")
print("Runtime module:", Interpreter.__module__)

import time
import cv2

MODEL_DIR = Path("/home/xilinx/jupyter_notebooks/models/ssd_mobilenet_v1")
MODEL_PATH = MODEL_DIR / "detect.tflite"
LABEL_PATH = MODEL_DIR / "labelmap.txt"
CONFIDENCE_THRESHOLD = 0.5
for path in (MODEL_PATH, LABEL_PATH):
    if not path.is_file():
        raise FileNotFoundError(f"Fichier introuvable sur le PYNQ : {path}")
print("Model path:", MODEL_PATH)
interpreter = None
try:
    interpreter = Interpreter(model_path=str(MODEL_PATH))
    interpreter.allocate_tensors()
except Exception:
    interpreter = None
    print("Model loading failed. Verifier le fichier TFLite CPU, les operateurs et la memoire.")
    raise

print("Model loading: OK")
print("Tensor allocation: OK")
if interpreter is None:
    raise RuntimeError("Charger et allouer le modele avant d'inspecter ses tenseurs.")

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

for group, details in [("Input", input_details), ("Output", output_details)]:
    print(f"\n{group} tensors: {len(details)}")
    for position, tensor in enumerate(details):
        print(f"\n{group} tensor #{position} (index={tensor['index']})")
        print("  name:", tensor["name"])
        print("  shape:", tensor["shape"].tolist())
        print("  shape signature:", tensor.get("shape_signature"))
        print("  dtype:", np.dtype(tensor["dtype"]).name)
        print("  quantization:", tensor["quantization"])
        print("  quantization parameters:", tensor["quantization_parameters"])

expected_shapes = [(1, 10, 4), (1, 10), (1, 10), (1,)]
if len(output_details) != 4:
    raise ValueError("Quatre sorties SSD sont attendues.")
for detail, expected in zip(output_details, expected_shapes):
    if tuple(detail["shape"]) != expected:
        raise ValueError(f"Output shape inattendue pour {detail['name']}: {detail['shape']}")


if not LABEL_PATH.is_file():
    raise FileNotFoundError(f"Label map introuvable sur le PYNQ : {LABEL_PATH}")
labels = [line.strip() for line in LABEL_PATH.read_text(encoding="utf-8-sig").splitlines()]
if labels and labels[0] == "???":
    labels = labels[1:]
if not labels or labels[0] != "person":
    raise ValueError("Label map incompatible : la classe 0 doit etre person.")
if not 0 <= CONFIDENCE_THRESHOLD <= 1:
    raise ValueError("Le seuil doit etre compris entre 0 et 1.")



NumPy import: OK — 1.21.5
Interpreter import: OK
Runtime module: tflite_runtime.interpreter
Model path: /home/xilinx/jupyter_notebooks/models/ssd_mobilenet_v1/detect.tflite
Model loading: OK
Tensor allocation: OK

Input tensors: 1

Input tensor #0 (index=175)
  name: normalized_input_image_tensor
  shape: [1, 300, 300, 3]
  shape signature: [  1 300 300   3]
  dtype: uint8
  quantization: (0.0078125, 128)
  quantization parameters: {'scales': array([0.0078125], dtype=float32), 'zero_points': array([128]), 'quantized_dimension': 0}

Output tensors: 4

Output tensor #0 (index=167)
  name: TFLite_Detection_PostProcess
  shape: [1, 10, 4]
  shape signature: [ 1 10  4]
  dtype: float32
  quantization: (0.0, 0)
  quantization parameters: {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}

Output tensor #1 (index=168)
  name: TFLite_Detection_PostProcess:1
  shape: [1, 10]
  shape signature: [ 1 10]
  dtype: float32
  quantization: (0.0, 0)
 

## 3. Camera configuration

La capture utilise OpenCV/V4L2, sans modifier le format ni optimiser la camera. Arreter toute autre utilisation de /dev/video0 sur le PYNQ avant ce test pour eviter un acces concurrent. La boucle est bornee ; une interruption Jupyter normale passe aussi par finally.


In [3]:
CAMERA_DEVICE = "/dev/video0"
MAX_INFERENCES = 10
if not isinstance(MAX_INFERENCES, int) or isinstance(MAX_INFERENCES, bool) or MAX_INFERENCES <= 0:
    raise ValueError("MAX_INFERENCES doit etre un entier strictement positif.")
print("Camera device:", CAMERA_DEVICE)
print("Maximum loop inferences:", MAX_INFERENCES)


Camera device: /dev/video0
Maximum loop inferences: 10


## 4. Single camera frame test

Capturer et valider une seule frame avant toute boucle. La camera est liberee immediatement ; la frame reste en memoire pour les etapes 5 et 6. La boucle rouvrira ensuite le meme peripherique.


In [4]:
single_frame_bgr = None
cap = cv2.VideoCapture(CAMERA_DEVICE, cv2.CAP_V4L2)
try:
    print("Camera opened:", cap.isOpened())
    if not cap.isOpened():
        raise RuntimeError(f"Impossible d'ouvrir {CAMERA_DEVICE} avec V4L2")
    success, frame = cap.read()
    print("Frame read successfully:", success and frame is not None)
    if not success or frame is None or frame.size == 0:
        raise RuntimeError("La camera n'a retourne aucune frame valide.")
    if frame.ndim != 3 or frame.shape[2] != 3 or frame.dtype != np.uint8:
        raise ValueError(f"Frame BGR uint8 attendue : {frame.shape}, {frame.dtype}")
    single_frame_bgr = frame.copy()
    print("Frame width/height:", frame.shape[1], frame.shape[0])
    print("Frame dtype:", frame.dtype)
finally:
    cap.release()
    print("Camera released")


Camera opened: True
Frame read successfully: True
Frame width/height: 640 480
Frame dtype: uint8
Camera released


## 5. Frame preprocessing

La fonction reprend le preprocessing de la baseline : BGR → RGB, resize 300×300 avec INTER_LINEAR, uint8 sans normalisation float, puis batch [1,300,300,3]. Elle sera reutilisee dans la boucle.


In [5]:
def preprocess_frame(image_bgr):
    if len(input_details) != 1:
        raise ValueError("Un seul tenseur d'entree est attendu.")
    if tuple(input_details[0]["shape"]) != (1, 300, 300, 3):
        raise ValueError(f"Input shape inattendue : {input_details[0]['shape']}")
    if np.dtype(input_details[0]["dtype"]) != np.dtype(np.uint8):
        raise TypeError("Le modele doit attendre des pixels uint8.")
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    resized_rgb = cv2.resize(image_rgb, (300, 300), interpolation=cv2.INTER_LINEAR)
    input_tensor = np.expand_dims(resized_rgb, axis=0)
    assert input_tensor.shape == (1, 300, 300, 3)
    assert input_tensor.dtype == np.uint8
    return input_tensor

input_tensor = preprocess_frame(single_frame_bgr)
print("Input shape:", input_tensor.shape)
print("Input dtype:", input_tensor.dtype)


Input shape: (1, 300, 300, 3)
Input dtype: uint8


## 6. Single-frame inference

Tester d'abord la frame conservee. Le chronometrage couvre seulement invoke(). Les sorties sont lues via output_details ; filtrage person au seuil 0.5 et conversion des boxes en pixels identiques a la baseline. Les petites fonctions evitent de dupliquer cette logique dans la boucle.


In [6]:
def infer_persons(input_tensor, original_height, original_width):
    interpreter.set_tensor(input_details[0]["index"], input_tensor)
    start_time = time.perf_counter()
    interpreter.invoke()
    inference_time_ms = (time.perf_counter() - start_time) * 1000
    
    boxes = interpreter.get_tensor(output_details[0]["index"])[0]
    classes = interpreter.get_tensor(output_details[1]["index"])[0]
    scores = interpreter.get_tensor(output_details[2]["index"])[0]
    count_value = float(interpreter.get_tensor(output_details[3]["index"])[0])
    if not np.isfinite(count_value) or not count_value.is_integer():
        raise ValueError(f"Nombre de detections invalide : {count_value}")
    num_detections = int(count_value)
    if not 0 <= num_detections <= min(len(boxes), len(classes), len(scores)):
        raise ValueError(f"Nombre de detections hors limites : {num_detections}")
    
    persons = []
    for i in range(num_detections):
        class_value = float(classes[i])
        score = float(scores[i])
        if not np.isfinite(class_value) or not class_value.is_integer():
            continue
        class_id = int(class_value)
        if not 0 <= class_id < len(labels):
            continue
        label = labels[class_id]
        if label != "person" or not np.isfinite(score) or score < CONFIDENCE_THRESHOLD:
            continue
        if not np.all(np.isfinite(boxes[i])):
            continue
        ymin, xmin, ymax, xmax = np.clip(boxes[i], 0.0, 1.0)
        x1 = int(np.floor(xmin * original_width))
        y1 = int(np.floor(ymin * original_height))
        x2 = int(np.ceil(xmax * original_width))
        y2 = int(np.ceil(ymax * original_height))
        if x2 <= x1 or y2 <= y1:
            continue
        persons.append({
            "label": label, "confidence": score,
            "bbox": {"x1": x1, "y1": y1, "x2": x2, "y2": y2},
        })
    return inference_time_ms, num_detections, persons


def print_result(number, elapsed_ms, count, persons):
    print(f"\nInference number: {number}")
    print(f"Inference time: {elapsed_ms:.2f} ms")
    print(f"Total detections returned: {count}")
    print(f"Persons detected: {len(persons)}")
    for person_number, person in enumerate(persons, start=1):
        box = person["bbox"]
        print(f"Person {person_number}: {person['label']}, confidence={person['confidence']:.1%}")
        print(f"bbox: x1={box['x1']}, y1={box['y1']}, x2={box['x2']}, y2={box['y2']}")

single_frame_validated = False
single_time_ms, single_count, single_persons = infer_persons(
    input_tensor, *single_frame_bgr.shape[:2]
)
print_result("single-frame test", single_time_ms, single_count, single_persons)
single_frame_validated = True



Inference number: single-frame test
Inference time: 1047.34 ms
Total detections returned: 10
Persons detected: 0


## 7. Continuous detection loop

Sequence simple : capture → preprocessing → invoke → lecture des sorties → filtrage → affichage texte. Une absence de personne est un resultat valide. Une lecture camera invalide arrete la cellule avec une erreur explicite. La camera est liberee dans tous les chemins de sortie Python, y compris KeyboardInterrupt. Aucun thread ni processus supplementaire n'est cree par ce notebook.


In [7]:
if not single_frame_validated:
    raise RuntimeError("Valider l'inference sur une frame avant la boucle.")
if not isinstance(MAX_INFERENCES, int) or isinstance(MAX_INFERENCES, bool) or MAX_INFERENCES <= 0:
    raise ValueError("MAX_INFERENCES doit etre un entier strictement positif.")

inference_times_ms = []
loop_start = time.perf_counter()
cap = cv2.VideoCapture(CAMERA_DEVICE, cv2.CAP_V4L2)
try:
    print("Camera opened:", cap.isOpened())
    if not cap.isOpened():
        raise RuntimeError(f"Impossible d'ouvrir {CAMERA_DEVICE} avec V4L2")
    for number in range(1, MAX_INFERENCES + 1):
        success, frame_bgr = cap.read()
        if not success or frame_bgr is None or frame_bgr.size == 0:
            raise RuntimeError(f"Lecture camera invalide avant l'inference {number}")
        if frame_bgr.ndim != 3 or frame_bgr.shape[2] != 3 or frame_bgr.dtype != np.uint8:
            raise ValueError("Frame camera BGR uint8 attendue")
        frame_tensor = preprocess_frame(frame_bgr)
        elapsed_ms, count, persons = infer_persons(frame_tensor, *frame_bgr.shape[:2])
        inference_times_ms.append(elapsed_ms)
        print_result(number, elapsed_ms, count, persons)
except KeyboardInterrupt:
    print("Boucle interrompue par l'utilisateur.")
finally:
    cap.release()
    loop_elapsed_seconds = time.perf_counter() - loop_start
    print("Camera released")


Camera opened: True

Inference number: 1
Inference time: 1163.08 ms
Total detections returned: 10
Persons detected: 0

Inference number: 2
Inference time: 1159.25 ms
Total detections returned: 10
Persons detected: 0

Inference number: 3
Inference time: 1213.61 ms
Total detections returned: 10
Persons detected: 0

Inference number: 4
Inference time: 1251.85 ms
Total detections returned: 10
Persons detected: 0

Inference number: 5
Inference time: 1166.21 ms
Total detections returned: 10
Persons detected: 0

Inference number: 6
Inference time: 1149.91 ms
Total detections returned: 10
Persons detected: 0

Inference number: 7
Inference time: 1158.42 ms
Total detections returned: 10
Persons detected: 0

Inference number: 8
Inference time: 1153.94 ms
Total detections returned: 10
Persons detected: 0

Inference number: 9
Inference time: 1438.92 ms
Total detections returned: 10
Persons detected: 0

Inference number: 10
Inference time: 1166.80 ms
Total detections returned: 10
Persons detected: 0

## 8. Performance summary

Statistiques des inferences terminees dans la boucle uniquement, hors test initial. Approximate inference FPS = 1000 / temps moyen de invoke(), ce n'est pas le debit camera ni une promesse temps reel. Le debit global inclut capture, preprocessing et affichage texte. Cette cellule peut aussi etre executee apres une interruption ou une erreur de la boucle pour afficher les mesures deja collectees.


In [8]:
total_inference_count = len(inference_times_ms)
print("Total inference count:", total_inference_count)
if inference_times_ms:
    average_ms = sum(inference_times_ms) / total_inference_count
    print(f"Average inference time: {average_ms:.2f} ms")
    print(f"Min inference time: {min(inference_times_ms):.2f} ms")
    print(f"Max inference time: {max(inference_times_ms):.2f} ms")
    print(f"Approximate inference FPS: {1000 / average_ms:.3f}" if average_ms > 0 else "Approximate inference FPS: unavailable")
    if loop_elapsed_seconds > 0:
        print(f"Overall loop FPS: {total_inference_count / loop_elapsed_seconds:.3f}")
else:
    print("Average/min/max inference time: unavailable (no completed inference)")
    print("Approximate inference FPS: unavailable")


Total inference count: 10
Average inference time: 1202.20 ms
Min inference time: 1149.91 ms
Max inference time: 1438.92 ms
Approximate inference FPS: 0.832
Overall loop FPS: 0.751
